# TVL Distribution Hourly Evolution

Notebook de visualisation pour suivre l'evolution de la distribution TVL hourly recentree autour du tick courant.

Objectifs:
- visualiser la distribution recentree heure par heure
- jouer l'evolution avec un slider + autoplay
- voir la forme globale via une heatmap temps x tick relatif
- suivre les metriques de recentrage (`current_bin_share_visible`, `share_overlapping_100_ticks`, `weighted_std_relative_tick`)
- deriver des vues daily et weekly en prenant le dernier snapshot horaire de chaque periode



## Prerequis

Le helper `hourly` reste utile pour la vue fine intra-day, mais les vues `daily` et `weekly` lisent maintenant un pipeline `daily` direct, beaucoup plus economique que `hourly -> aggregate`.

Build recommande:

```bash
cd /home/gharbi/defind-analytics/dbt/defind_analytics
../../.venv/bin/dbt run --target local --select int_hourly_snapshot_blocks int_hourly_tick_liquidity_net
../../.venv/bin/dbt run --target local --select mart_tvl_distribution_hourly_centered mart_tvl_distribution_hourly_metrics --vars '{"hourly_distribution_start_hour":"2026-04-01 00:00:00","hourly_distribution_end_hour":"2026-04-07 23:00:00","hourly_distribution_window_ticks":4000}'
../../.venv/bin/dbt run --target local --select int_daily_snapshot_blocks int_daily_tick_liquidity_net mart_tvl_distribution_daily_centered mart_tvl_distribution_daily_metrics --vars '{"daily_distribution_start_date":"2025-01-01","daily_distribution_end_date":"2026-04-07","daily_distribution_window_ticks":4000}'
```

Le notebook lit ensuite directement les tables materialisees.


In [47]:
import os
from textwrap import dedent

import clickhouse_connect

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "pandas is required. Run: %pip install pandas clickhouse-connect plotly jupyterlab"
    ) from exc

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ModuleNotFoundError as exc:
    PLOTLY_AVAILABLE = False
    raise ModuleNotFoundError(
        "plotly is required. Run: %pip install plotly"
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)


In [48]:
# Defaults local-first pour clickhouse_local.
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT", "8124"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "1234")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "analytics")

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

print(f"Connected to ClickHouse database={CLICKHOUSE_DATABASE} on {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}")


Connected to ClickHouse database=analytics on localhost:8124


In [49]:
TIME_COLUMNS = ["date", "hour", "snapshot_block_time", "block_time", "deployment_ts", "day", "week_start"]

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    for column in TIME_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column])
    return df

def query_df(sql: str, parameters: dict | None = None) -> pd.DataFrame:
    df = client.query_df(sql, parameters=parameters or {})
    df = normalize_df(df)
    print(f"{len(df):,} rows")
    return df

def query_scalar(sql: str, parameters: dict | None = None):
    return client.query(sql, parameters=parameters or {}).first_item

def plot_guard():
    if not PLOTLY_AVAILABLE:
        raise RuntimeError("plotly is not available")


## 1. Coverage and pool selection


In [50]:
coverage_df = query_df(dedent("""
    SELECT
        pool,
        min(hour) AS min_hour,
        max(hour) AS max_hour,
        count() AS hourly_points,
        avg(visible_tvl_token0) AS avg_visible_tvl_token0,
        avg(weighted_std_relative_tick) AS avg_weighted_std_relative_tick
    FROM mart_tvl_distribution_hourly_metrics
    GROUP BY pool
    ORDER BY max_hour DESC, pool
"""))
coverage_df


1 rows


,pool,min_hour,max_hour,hourly_points,avg_visible_tvl_token0,avg_weighted_std_relative_tick
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-06 11:00:00,132,7.524318e+07,1324.282128


In [51]:

# Ajuste ces parametres si besoin.
POOL = coverage_df.iloc[0]["pool"] if not coverage_df.empty else None
HOUR_FROM = coverage_df.iloc[0]["min_hour"] if not coverage_df.empty else None
HOUR_TO = coverage_df.iloc[0]["max_hour"] if not coverage_df.empty else None
RELATIVE_TICK_LIMIT = 4000
DISTRIBUTION_METRIC = "pct_visible_tvl"  # alternatives: tvl_token0, pct_visible_liquidity, liquidity
DAILY_LOOKBACK_DAYS = 90
WEEKLY_LOOKBACK_WEEKS = 52

print("POOL:", POOL)
print("HOUR_FROM:", HOUR_FROM)
print("HOUR_TO:", HOUR_TO)
print("RELATIVE_TICK_LIMIT:", RELATIVE_TICK_LIMIT)
print("DISTRIBUTION_METRIC:", DISTRIBUTION_METRIC)
print("DAILY_LOOKBACK_DAYS:", DAILY_LOOKBACK_DAYS)
print("WEEKLY_LOOKBACK_WEEKS:", WEEKLY_LOOKBACK_WEEKS)


POOL: 0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640
HOUR_FROM: 2026-04-01 00:00:00
HOUR_TO: 2026-04-06 11:00:00
RELATIVE_TICK_LIMIT: 4000
DISTRIBUTION_METRIC: pct_visible_tvl
DAILY_LOOKBACK_DAYS: 90
WEEKLY_LOOKBACK_WEEKS: 52


## 2. Load hourly centered distribution and metrics


In [52]:
params = {
    "pool": POOL,
    "hour_from": HOUR_FROM,
    "hour_to": HOUR_TO,
    "relative_tick_limit": RELATIVE_TICK_LIMIT,
}

distribution_df = query_df(dedent("""
    SELECT
        pool,
        date,
        hour,
        snapshot_block_number,
        snapshot_log_index,
        snapshot_block_time,
        current_tick,
        price_token1_in_token0,
        tick_lower,
        tick_upper,
        relative_tick_lower,
        relative_tick_upper,
        relative_tick_mid,
        liquidity,
        is_active_bin,
        amount0,
        amount1,
        tvl_token0,
        pct_visible_liquidity,
        pct_visible_tvl
    FROM mart_tvl_distribution_hourly_centered
    WHERE pool = %(pool)s
      AND hour >= %(hour_from)s
      AND hour <= %(hour_to)s
      AND abs(relative_tick_mid) <= %(relative_tick_limit)s
    ORDER BY hour, relative_tick_mid
"""), params)

metrics_df = query_df(dedent("""
    SELECT
        pool,
        date,
        hour,
        snapshot_block_number,
        snapshot_log_index,
        snapshot_block_time,
        current_tick,
        price_token1_in_token0,
        visible_tvl_token0,
        current_bin_tvl_token0,
        current_bin_liquidity,
        current_bin_share_visible,
        share_overlapping_50_ticks,
        share_overlapping_100_ticks,
        share_overlapping_250_ticks,
        share_overlapping_500_ticks,
        weighted_mean_relative_tick,
        weighted_abs_relative_tick,
        weighted_std_relative_tick
    FROM mart_tvl_distribution_hourly_metrics
    WHERE pool = %(pool)s
      AND hour >= %(hour_from)s
      AND hour <= %(hour_to)s
    ORDER BY hour
"""), params)

distribution_df["hour_label"] = distribution_df["hour"].dt.strftime("%Y-%m-%d %H:%M")
metrics_df["hour_label"] = metrics_df["hour"].dt.strftime("%Y-%m-%d %H:%M")

display(distribution_df.head())
display(metrics_df.head())


105,611 rows
132 rows


,pool,date,hour,snapshot_block_number,snapshot_log_index,snapshot_block_time,current_tick,price_token1_in_token0,tick_lower,tick_upper,relative_tick_lower,relative_tick_upper,relative_tick_mid,liquidity,is_active_bin,amount0,amount1,tvl_token0,pct_visible_liquidity,pct_visible_tvl,hour_label
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01,24781326,882,2026-04-01 00:59:59,199760,2113.278344,195760,195770,-4000,-3990,-3995.0,1.343046e+18,0,0.0,11.962163,25279.380551,0.039289,0.033887,2026-04-01 00:00
1,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01,24781326,882,2026-04-01 00:59:59,199760,2113.278344,195770,195780,-3990,-3980,-3985.0,1.341759e+18,0,0.0,11.956677,25267.786677,0.039252,0.033872,2026-04-01 00:00
2,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01,24781326,882,2026-04-01 00:59:59,199760,2113.278344,195780,195790,-3980,-3970,-3975.0,1.343445e+18,0,0.0,11.977683,25312.177088,0.039301,0.033931,2026-04-01 00:00
3,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01,24781326,882,2026-04-01 00:59:59,199760,2113.278344,195790,195800,-3970,-3960,-3965.0,1.343450e+18,0,0.0,11.983720,25324.935036,0.039301,0.033948,2026-04-01 00:00
4,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01,24781326,882,2026-04-01 00:59:59,199760,2113.278344,195800,195810,-3960,-3950,-3955.0,1.343317e+18,0,0.0,11.988523,25335.086969,0.039297,0.033962,2026-04-01 00:00


,pool,date,hour,snapshot_block_number,snapshot_log_index,snapshot_block_time,current_tick,price_token1_in_token0,visible_tvl_token0,current_bin_tvl_token0,current_bin_liquidity,current_bin_share_visible,share_overlapping_50_ticks,share_overlapping_100_ticks,share_overlapping_250_ticks,share_overlapping_500_ticks,weighted_mean_relative_tick,weighted_abs_relative_tick,weighted_std_relative_tick,hour_label
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01 00:00:00,24781326,882,2026-04-01 00:59:59,199760,2113.278344,7.459810e+07,295010.615423,1.283851e+19,0.003955,0.039649,0.079498,0.198199,0.371936,281.944881,1015.000237,1346.923167,2026-04-01 00:00
1,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01 01:00:00,24781622,876,2026-04-01 01:59:11,199859,2092.414254,7.426342e+07,301952.995632,1.320580e+19,0.004066,0.044655,0.084722,0.203449,0.405519,208.653851,983.596045,1336.863190,2026-04-01 01:00
2,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01 02:00:00,24781918,73,2026-04-01 02:58:59,199882,2087.531140,7.417121e+07,301951.247560,1.322016e+19,0.004071,0.044701,0.085094,0.203731,0.414054,193.422843,976.768307,1333.531679,2026-04-01 02:00
3,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01 03:00:00,24782218,563,2026-04-01 03:59:35,199816,2101.399898,7.441713e+07,301496.040177,1.315646e+19,0.004051,0.044301,0.084192,0.203044,0.394175,240.673707,996.699584,1341.603852,2026-04-01 03:00
4,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,2026-04-01 04:00:00,24782520,1018,2026-04-01 04:59:59,199773,2110.451848,7.456093e+07,294834.646258,1.283810e+19,0.003954,0.044399,0.084751,0.202650,0.380189,272.487624,1010.916290,1346.606563,2026-04-01 04:00



## 3. Direct daily views and weekly last-day snapshots


In [53]:

daily_to = pd.Timestamp(HOUR_TO).floor("D") if HOUR_TO is not None else None
max_daily_history_days = max(DAILY_LOOKBACK_DAYS + 14, WEEKLY_LOOKBACK_WEEKS * 7 + 14)
daily_from = daily_to - pd.Timedelta(days=max_daily_history_days) if daily_to is not None else None

daily_params = {
    "pool": POOL,
    "date_from": daily_from.strftime("%Y-%m-%d") if daily_from is not None else None,
    "date_to": daily_to.strftime("%Y-%m-%d") if daily_to is not None else None,
    "relative_tick_limit": RELATIVE_TICK_LIMIT,
}

full_daily_distribution_df = query_df(dedent("""
    SELECT
        pool,
        date,
        snapshot_block_number,
        snapshot_log_index,
        snapshot_block_time,
        current_tick,
        price_token1_in_token0,
        tick_lower,
        tick_upper,
        relative_tick_lower,
        relative_tick_upper,
        relative_tick_mid,
        liquidity,
        is_active_bin,
        amount0,
        amount1,
        tvl_token0,
        pct_visible_liquidity,
        pct_visible_tvl
    FROM mart_tvl_distribution_daily_centered
    WHERE pool = %(pool)s
      AND date >= %(date_from)s
      AND date <= %(date_to)s
      AND abs(relative_tick_mid) <= %(relative_tick_limit)s
    ORDER BY date, relative_tick_mid
"""), daily_params)

full_daily_metrics_df = query_df(dedent("""
    SELECT
        pool,
        date,
        snapshot_block_number,
        snapshot_log_index,
        snapshot_block_time,
        current_tick,
        price_token1_in_token0,
        visible_tvl_token0,
        current_bin_tvl_token0,
        current_bin_liquidity,
        current_bin_share_visible,
        share_overlapping_50_ticks,
        share_overlapping_100_ticks,
        share_overlapping_250_ticks,
        share_overlapping_500_ticks,
        weighted_mean_relative_tick,
        weighted_abs_relative_tick,
        weighted_std_relative_tick
    FROM mart_tvl_distribution_daily_metrics
    WHERE pool = %(pool)s
      AND date >= %(date_from)s
      AND date <= %(date_to)s
    ORDER BY date
"""), daily_params)

full_daily_distribution_df["period_label"] = full_daily_distribution_df["date"].dt.strftime("%Y-%m-%d")
full_daily_metrics_df = full_daily_metrics_df.sort_values("date").copy()
full_daily_metrics_df["day"] = full_daily_metrics_df["date"]
full_daily_metrics_df["period_label"] = full_daily_metrics_df["date"].dt.strftime("%Y-%m-%d")

# Daily animation stays focused on the latest 90 days.
daily_metrics_df = full_daily_metrics_df.tail(DAILY_LOOKBACK_DAYS).copy()
daily_distribution_df = full_daily_distribution_df.merge(
    daily_metrics_df[["pool", "date"]].drop_duplicates(),
    on=["pool", "date"],
    how="inner",
)

# Weekly snapshots are derived from the full daily history, not from the 90-day tail.
weekly_metrics_df = (
    full_daily_metrics_df
    .assign(week_start=lambda df: df["date"].dt.to_period("W-SUN").apply(lambda p: p.start_time))
    .sort_values("date")
    .groupby(["pool", "week_start"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)
weekly_metrics_df = weekly_metrics_df.tail(WEEKLY_LOOKBACK_WEEKS).copy()
weekly_metrics_df["period_label"] = weekly_metrics_df["week_start"].dt.strftime("Week of %Y-%m-%d")

weekly_distribution_df = full_daily_distribution_df.merge(
    weekly_metrics_df[["pool", "date", "week_start"]],
    on=["pool", "date"],
    how="inner",
)
weekly_distribution_df["period_label"] = weekly_distribution_df["week_start"].dt.strftime("Week of %Y-%m-%d")

display(full_daily_metrics_df[["date", "current_tick", "share_overlapping_100_ticks", "weighted_std_relative_tick"]].tail())
display(weekly_metrics_df[["week_start", "date", "current_tick", "share_overlapping_100_ticks", "weighted_std_relative_tick"]].tail())


301,078 rows
379 rows


,date,current_tick,share_overlapping_100_ticks,weighted_std_relative_tick
374,2026-04-02,200031,0.086777,1306.810662
375,2026-04-03,200047,0.086794,1309.761514
376,2026-04-04,199992,0.086824,1312.408715
377,2026-04-05,199778,0.084007,1339.474934
378,2026-04-06,199587,0.089528,1366.213519


,week_start,date,current_tick,share_overlapping_100_ticks,weighted_std_relative_tick
50,2026-03-09,2026-03-15,199459,0.071058,1408.698369
51,2026-03-16,2026-03-22,200048,0.082294,1314.013270
52,2026-03-23,2026-03-29,200394,0.096320,1285.635276
53,2026-03-30,2026-04-05,199778,0.084007,1339.474934
54,2026-04-06,2026-04-06,199587,0.089528,1366.213519


In [54]:

period_summary_df = pd.DataFrame([{
    "period": "daily_direct_full_history",
    "points": daily_metrics_df["date"].nunique(),
    "min_date": daily_metrics_df["date"].min(),
    "max_date": daily_metrics_df["date"].max(),
    "avg_share_overlapping_100_ticks": daily_metrics_df["share_overlapping_100_ticks"].mean(),
    "avg_weighted_std_relative_tick": daily_metrics_df["weighted_std_relative_tick"].mean(),
}, {
    "period": "weekly_last_day",
    "points": weekly_metrics_df["week_start"].nunique(),
    "min_week_start": weekly_metrics_df["week_start"].min(),
    "max_week_start": weekly_metrics_df["week_start"].max(),
    "min_snapshot_date": weekly_metrics_df["date"].min(),
    "max_snapshot_date": weekly_metrics_df["date"].max(),
    "avg_share_overlapping_100_ticks": weekly_metrics_df["share_overlapping_100_ticks"].mean(),
    "avg_weighted_std_relative_tick": weekly_metrics_df["weighted_std_relative_tick"].mean(),
}])
period_summary_df


,period,points,min_date,max_date,avg_share_overlapping_100_ticks,avg_weighted_std_relative_tick,min_week_start,max_week_start,min_snapshot_date,max_snapshot_date
0,daily_direct_full_history,90,2026-01-07,2026-04-06,0.067261,1591.336334,NaT,NaT,NaT,NaT
1,weekly_last_day,52,NaT,NaT,0.056329,1794.412911,2025-04-14,2026-04-06,2025-04-20,2026-04-06


## 4. Quick QA summary


In [55]:
summary_df = pd.DataFrame([{
    "pool": POOL,
    "hours": metrics_df["hour"].nunique(),
    "bins": len(distribution_df),
    "min_hour": metrics_df["hour"].min(),
    "max_hour": metrics_df["hour"].max(),
    "avg_visible_tvl_token0": metrics_df["visible_tvl_token0"].mean(),
    "avg_share_overlapping_100_ticks": metrics_df["share_overlapping_100_ticks"].mean(),
    "avg_weighted_std_relative_tick": metrics_df["weighted_std_relative_tick"].mean(),
}])
summary_df


,pool,hours,bins,min_hour,max_hour,avg_visible_tvl_token0,avg_share_overlapping_100_ticks,avg_weighted_std_relative_tick
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,132,105611,2026-04-01,2026-04-06 11:00:00,7.524318e+07,0.086994,1324.282128


## 5. Animated centered distribution

Ce graphique supporte deja les deux usages: slider manuel et autoplay via le bouton play.


In [56]:
plot_guard()

metric_title_map = {
    "pct_visible_tvl": "% visible TVL",
    "pct_visible_liquidity": "% visible liquidity",
    "tvl_token0": "TVL token0",
    "liquidity": "Liquidity",
}

distribution_plot_df = distribution_df.copy()
distribution_plot_df["active_bin"] = distribution_plot_df["is_active_bin"].map({1: "Active bin", 0: "Inactive bin"})

y_max = float(distribution_plot_df[DISTRIBUTION_METRIC].quantile(0.995)) if not distribution_plot_df.empty else 1.0
y_max = max(y_max, float(distribution_plot_df[DISTRIBUTION_METRIC].max())) if not distribution_plot_df.empty else 1.0

fig_animation = px.bar(
    distribution_plot_df,
    x="relative_tick_mid",
    y=DISTRIBUTION_METRIC,
    color="active_bin",
    animation_frame="hour_label",
    animation_group="relative_tick_mid",
    category_orders={"active_bin": ["Inactive bin", "Active bin"]},
    color_discrete_map={"Inactive bin": "#9ecae1", "Active bin": "#ef3b2c"},
    labels={
        "relative_tick_mid": "Tick relatif au current tick",
        DISTRIBUTION_METRIC: metric_title_map.get(DISTRIBUTION_METRIC, DISTRIBUTION_METRIC),
        "hour_label": "Hour",
    },
    title="Hourly centered TVL distribution around the current tick",
)
fig_animation.update_layout(
    bargap=0.0,
    height=560,
    legend_title_text="Bin type",
    xaxis_range=[-RELATIVE_TICK_LIMIT, RELATIVE_TICK_LIMIT],
    yaxis_range=[0, y_max * 1.05],
    transition={"duration": 0},
)
fig_animation.update_xaxes(showgrid=True, zeroline=True, zerolinewidth=1, zerolinecolor="#111111")
fig_animation.show()


## 6. Heatmap of the recentered distribution through time


In [57]:
plot_guard()

heatmap_metric = "pct_visible_tvl"
heatmap_df = (
    distribution_df
    .pivot_table(index="relative_tick_mid", columns="hour", values=heatmap_metric, aggfunc="sum", fill_value=0.0)
    .sort_index()
)

fig_heatmap = go.Figure(
    data=go.Heatmap(
        z=heatmap_df.values,
        x=heatmap_df.columns,
        y=heatmap_df.index,
        colorscale="YlOrRd",
        colorbar_title="% visible TVL",
    )
)
fig_heatmap.update_layout(
    title="Heatmap: hourly centered TVL distribution through time",
    xaxis_title="Hour",
    yaxis_title="Relative tick midpoint",
    height=620,
)
fig_heatmap.show()


## 7. Re-centering metrics over time


In [58]:
plot_guard()

fig_metrics = make_subplots(specs=[[{"secondary_y": True}]])

fig_metrics.add_trace(
    go.Scatter(
        x=metrics_df["hour"],
        y=metrics_df["current_bin_share_visible"],
        mode="lines",
        name="Current bin share visible",
        line=dict(color="#1f77b4", width=2),
    ),
    secondary_y=False,
)

fig_metrics.add_trace(
    go.Scatter(
        x=metrics_df["hour"],
        y=metrics_df["share_overlapping_100_ticks"],
        mode="lines",
        name="Share overlapping +/-100 ticks",
        line=dict(color="#2ca02c", width=2),
    ),
    secondary_y=False,
)

fig_metrics.add_trace(
    go.Scatter(
        x=metrics_df["hour"],
        y=metrics_df["weighted_std_relative_tick"],
        mode="lines",
        name="Weighted std relative tick",
        line=dict(color="#d62728", width=2),
    ),
    secondary_y=True,
)

fig_metrics.update_layout(
    title="Hourly re-centering metrics",
    height=520,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_metrics.update_xaxes(title_text="Hour")
fig_metrics.update_yaxes(title_text="Share of visible TVL", secondary_y=False)
fig_metrics.update_yaxes(title_text="Weighted std relative tick", secondary_y=True)
fig_metrics.show()



## 8. Full daily history and weekly visualizations


In [59]:

plot_guard()

full_daily_heatmap_df = (
    full_daily_distribution_df
    .pivot_table(index="relative_tick_mid", columns="date", values="pct_visible_tvl", aggfunc="sum", fill_value=0.0)
    .sort_index()
)

fig_full_daily_heatmap = go.Figure(
    data=go.Heatmap(
        z=full_daily_heatmap_df.values,
        x=full_daily_heatmap_df.columns,
        y=full_daily_heatmap_df.index,
        colorscale="YlGnBu",
        colorbar_title="% visible TVL",
    )
)
fig_full_daily_heatmap.update_layout(
    title="Full daily centered distribution heatmap",
    xaxis_title="Day",
    yaxis_title="Relative tick midpoint",
    height=560,
)
fig_full_daily_heatmap.show()

fig_full_daily_metrics = make_subplots(specs=[[{"secondary_y": True}]])
fig_full_daily_metrics.add_trace(go.Scatter(x=full_daily_metrics_df["date"], y=full_daily_metrics_df["share_overlapping_100_ticks"], mode="lines+markers", name="Daily share +/-100", line=dict(color="#2ca02c", width=2)), secondary_y=False)
fig_full_daily_metrics.add_trace(go.Scatter(x=full_daily_metrics_df["date"], y=full_daily_metrics_df["weighted_std_relative_tick"], mode="lines+markers", name="Daily weighted std", line=dict(color="#d62728", width=2)), secondary_y=True)
fig_full_daily_metrics.update_layout(title="Full daily re-centering metrics", height=480)
fig_full_daily_metrics.update_xaxes(title_text="Day")
fig_full_daily_metrics.update_yaxes(title_text="Share of visible TVL", secondary_y=False)
fig_full_daily_metrics.update_yaxes(title_text="Weighted std relative tick", secondary_y=True)
fig_full_daily_metrics.show()

weekly_heatmap_df = (
    weekly_distribution_df
    .pivot_table(index="relative_tick_mid", columns="week_start", values="pct_visible_tvl", aggfunc="sum", fill_value=0.0)
    .sort_index()
)

fig_weekly_heatmap = go.Figure(
    data=go.Heatmap(
        z=weekly_heatmap_df.values,
        x=weekly_heatmap_df.columns,
        y=weekly_heatmap_df.index,
        colorscale="PuBuGn",
        colorbar_title="% visible TVL",
    )
)
fig_weekly_heatmap.update_layout(
    title="Weekly last-day centered distribution heatmap",
    xaxis_title="Week start",
    yaxis_title="Relative tick midpoint",
    height=560,
)
fig_weekly_heatmap.show()

fig_weekly_metrics = make_subplots(specs=[[{"secondary_y": True}]])
fig_weekly_metrics.add_trace(go.Scatter(x=weekly_metrics_df["week_start"], y=weekly_metrics_df["share_overlapping_100_ticks"], mode="lines+markers", name="Weekly share +/-100", line=dict(color="#17becf", width=2)), secondary_y=False)
fig_weekly_metrics.add_trace(go.Scatter(x=weekly_metrics_df["week_start"], y=weekly_metrics_df["weighted_std_relative_tick"], mode="lines+markers", name="Weekly weighted std", line=dict(color="#9467bd", width=2)), secondary_y=True)
fig_weekly_metrics.update_layout(title="Weekly last-day re-centering metrics", height=480)
fig_weekly_metrics.update_xaxes(title_text="Week start")
fig_weekly_metrics.update_yaxes(title_text="Share of visible TVL", secondary_y=False)
fig_weekly_metrics.update_yaxes(title_text="Weighted std relative tick", secondary_y=True)
fig_weekly_metrics.show()



## 9. Animated daily (90d) and weekly (52w) distributions


In [60]:

plot_guard()

def build_distribution_animation(df: pd.DataFrame, frame_col: str, title: str, x_label: str):
    if df.empty:
        print(f"No rows available for {title}")
        return None
    local_df = df.copy()
    local_df["active_bin"] = local_df["is_active_bin"].map({1: "Active bin", 0: "Inactive bin"})
    y_max = float(local_df["pct_visible_tvl"].quantile(0.995)) if not local_df.empty else 1.0
    y_max = max(y_max, float(local_df["pct_visible_tvl"].max())) if not local_df.empty else 1.0
    fig = px.bar(
        local_df,
        x="relative_tick_mid",
        y="pct_visible_tvl",
        color="active_bin",
        animation_frame=frame_col,
        animation_group="relative_tick_mid",
        category_orders={"active_bin": ["Inactive bin", "Active bin"]},
        color_discrete_map={"Inactive bin": "#9ecae1", "Active bin": "#ef3b2c"},
        labels={
            "relative_tick_mid": "Tick relatif au current tick",
            "pct_visible_tvl": "% visible TVL",
            frame_col: x_label,
        },
        title=title,
    )
    fig.update_layout(
        bargap=0.0,
        height=560,
        legend_title_text="Bin type",
        xaxis_range=[-RELATIVE_TICK_LIMIT, RELATIVE_TICK_LIMIT],
        yaxis_range=[0, y_max * 1.05],
        transition={"duration": 0},
    )
    fig.update_xaxes(showgrid=True, zeroline=True, zerolinewidth=1, zerolinecolor="#111111")
    return fig

fig_daily_anim = build_distribution_animation(
    daily_distribution_df,
    frame_col="period_label",
    title=f"Daily centered distribution ({DAILY_LOOKBACK_DAYS} days)",
    x_label="Day",
)
if fig_daily_anim is not None:
    fig_daily_anim.show()

fig_weekly_anim = build_distribution_animation(
    weekly_distribution_df,
    frame_col="period_label",
    title=f"Weekly last-day centered distribution ({WEEKLY_LOOKBACK_WEEKS} weeks)",
    x_label="Week",
)
if fig_weekly_anim is not None:
    fig_weekly_anim.show()



## 10. Animated daily (90d) and weekly (52w) metrics


In [61]:

plot_guard()

def build_metrics_animation(df: pd.DataFrame, time_col: str, frame_col: str, title: str):
    if df.empty:
        print(f"No rows available for {title}")
        return None
    frames = []
    ordered = df.sort_values(time_col).copy()
    base_periods = ordered[time_col].tolist()
    for current_period in base_periods:
        subset = ordered[ordered[time_col] <= current_period]
        frames.append(
            go.Frame(
                name=subset.iloc[-1][frame_col],
                data=[
                    go.Scatter(x=subset[time_col], y=subset["share_overlapping_100_ticks"], mode="lines+markers", line=dict(color="#2ca02c", width=2)),
                    go.Scatter(x=subset[time_col], y=subset["weighted_std_relative_tick"], mode="lines+markers", line=dict(color="#d62728", width=2), yaxis="y2"),
                ],
            )
        )
    fig = go.Figure(
        data=[
            go.Scatter(x=[], y=[], mode="lines+markers", name="Share overlapping +/-100 ticks", line=dict(color="#2ca02c", width=2)),
            go.Scatter(x=[], y=[], mode="lines+markers", name="Weighted std relative tick", line=dict(color="#d62728", width=2), yaxis="y2"),
        ],
        frames=frames,
    )
    fig.update_layout(
        title=title,
        xaxis=dict(title=str(time_col)),
        yaxis=dict(title="Share of visible TVL", range=[0, max(1.0, float(ordered["share_overlapping_100_ticks"].max()) * 1.05)]),
        yaxis2=dict(title="Weighted std relative tick", overlaying="y", side="right", range=[0, float(ordered["weighted_std_relative_tick"].max()) * 1.05]),
        height=500,
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "x": 1.02,
            "y": 1.12,
            "buttons": [
                {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": 220, "redraw": True}, "fromcurrent": True}]},
                {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]},
            ],
        }],
        sliders=[{
            "active": 0,
            "currentvalue": {"prefix": "Period: "},
            "steps": [
                {"label": frame.name, "method": "animate", "args": [[frame.name], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}}]}
                for frame in frames
            ],
        }],
    )
    if frames:
        fig.update(data=frames[0].data)
    return fig

fig_daily_metrics_anim = build_metrics_animation(
    daily_metrics_df,
    time_col="date",
    frame_col="period_label",
    title=f"Daily animated metrics ({DAILY_LOOKBACK_DAYS} days)",
)
if fig_daily_metrics_anim is not None:
    fig_daily_metrics_anim.show()

fig_weekly_metrics_anim = build_metrics_animation(
    weekly_metrics_df,
    time_col="week_start",
    frame_col="period_label",
    title=f"Weekly last-day animated metrics ({WEEKLY_LOOKBACK_WEEKS} weeks)",
)
if fig_weekly_metrics_anim is not None:
    fig_weekly_metrics_anim.show()


## 11. Optional: animated hourly metrics with autoplay

Ce graphique evolue automatiquement et montre la trajectoire de la concentration et de la dispersion dans le temps.


In [62]:
plot_guard()

frames = []
base_hours = metrics_df["hour"].sort_values().tolist()

for current_hour in base_hours:
    subset = metrics_df[metrics_df["hour"] <= current_hour]
    frames.append(
        go.Frame(
            name=current_hour.strftime("%Y-%m-%d %H:%M"),
            data=[
                go.Scatter(x=subset["hour"], y=subset["share_overlapping_100_ticks"], mode="lines", line=dict(color="#2ca02c", width=2)),
                go.Scatter(x=subset["hour"], y=subset["weighted_std_relative_tick"], mode="lines", line=dict(color="#d62728", width=2), yaxis="y2"),
            ],
        )
    )

fig_metrics_anim = go.Figure(
    data=[
        go.Scatter(x=[], y=[], mode="lines", name="Share overlapping +/-100 ticks", line=dict(color="#2ca02c", width=2)),
        go.Scatter(x=[], y=[], mode="lines", name="Weighted std relative tick", line=dict(color="#d62728", width=2), yaxis="y2"),
    ],
    frames=frames,
)

fig_metrics_anim.update_layout(
    title="Animated hourly re-centering metrics",
    xaxis=dict(title="Hour"),
    yaxis=dict(title="Share of visible TVL", range=[0, max(1.0, float(metrics_df["share_overlapping_100_ticks"].max()) * 1.05)]),
    yaxis2=dict(title="Weighted std relative tick", overlaying="y", side="right", range=[0, float(metrics_df["weighted_std_relative_tick"].max()) * 1.05]),
    height=520,
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "x": 1.02,
        "y": 1.12,
        "buttons": [
            {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": 160, "redraw": True}, "fromcurrent": True}]},
            {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]},
        ],
    }],
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "Hour: "},
        "steps": [
            {"label": frame.name, "method": "animate", "args": [[frame.name], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}}]}
            for frame in frames
        ],
    }],
)

if frames:
    fig_metrics_anim.update(data=frames[0].data)

fig_metrics_anim.show()


## 12. Last observed hours


In [63]:
latest_hours_preview = metrics_df.tail(10).copy()
latest_hours_preview[[
    "hour",
    "current_tick",
    "visible_tvl_token0",
    "current_bin_share_visible",
    "share_overlapping_100_ticks",
    "weighted_std_relative_tick",
]]


,hour,current_tick,visible_tvl_token0,current_bin_share_visible,share_overlapping_100_ticks,weighted_std_relative_tick
122,2026-04-06 02:00:00,199669,7.634001e+07,0.003905,0.083404,1360.821733
123,2026-04-06 03:00:00,199691,7.625627e+07,0.004061,0.083702,1356.906877
124,2026-04-06 04:00:00,199679,7.624846e+07,0.003903,0.083472,1359.876180
125,2026-04-06 05:00:00,199680,7.621486e+07,0.004064,0.079568,1357.640537
126,2026-04-06 06:00:00,199692,7.678948e+07,0.004275,0.085802,1352.507205
127,2026-04-06 07:00:00,199656,7.689747e+07,0.004261,0.086846,1358.043072
128,2026-04-06 08:00:00,199640,7.689554e+07,0.004221,0.083147,1358.953058
129,2026-04-06 09:00:00,199581,7.711128e+07,0.004363,0.089585,1366.193211
130,2026-04-06 10:00:00,199576,7.721976e+07,0.004362,0.089441,1367.045647
131,2026-04-06 11:00:00,199587,7.719449e+07,0.004360,0.089528,1366.213519
